# ISOM 835 · Session 6 — From Scores to Decisions: Thresholds, Costs & Calibration
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Oct 26 · Prof. Hasan Arslan**

A model produces a probability; a business produces a decision. Tonight: the expected-value framework, profit curves, `TunedThresholdClassifierCV`, calibration, lift and gain charts, and targeting under a budget — all on Telco churn.

> **Frame the decision.** *Unit:* one customer · *Score:* P(churn next month) · *Action:* send a $50 retention offer or not · *Value of a caught churner:* $450 in expectation (a $1,500 customer × 30% acceptance) · **Break-even:** p > 50 / (50 + 450) = 0.10.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, confusion_matrix, roc_auc_score, brier_score_loss
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL)
X = pd.get_dummies(df[['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']], drop_first=True)
y = (df['Churn'] == 'Yes').astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)
OFFER, SAVE = 50, 450

## 1. Who chose 0.5?

In [ ]:
base = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
p = base.predict_proba(X_te)[:, 1]
def profit(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return SAVE * tp - OFFER * (tp + fp)
print('confusion at 0.5:'); print(confusion_matrix(y_te, p >= 0.5))
print(f'profit at 0.5: ${profit(y_te, p >= 0.5):,}   AUC {roc_auc_score(y_te, p):.3f}')

## 2. The profit curve
Sweep the threshold, compute profit at each, plot. Accuracy peaks somewhere; profit peaks somewhere else. The business is paid in profit.

In [ ]:
ths = np.linspace(0.01, 0.99, 99)
prof = np.array([profit(y_te, p >= t) for t in ths]); acc = np.array([((p >= t) == y_te).mean() for t in ths])
best_t, best_acc_t = ths[prof.argmax()], ths[acc.argmax()]
fig, ax1 = plt.subplots(figsize=(7, 3.6)); ax1.plot(ths, prof, color='#7ee787', lw=2, label='profit'); ax1.axvline(best_t, color='#f5a524', ls='--'); ax1.set_xlabel('threshold'); ax1.set_ylabel('profit ($)')
ax2 = ax1.twinx(); ax2.plot(ths, acc, color='#7c8cff', lw=1.5, ls=':', label='accuracy'); ax2.set_ylabel('accuracy'); ax1.set_title(f'profit-optimal threshold {best_t:.2f}   accuracy-optimal {best_acc_t:.2f}'); plt.show()
print(f'profit at 0.50: ${prof[np.abs(ths-0.5).argmin()]:,}   at {best_t:.2f}: ${prof.max():,}   (+${prof.max()-prof[np.abs(ths-0.5).argmin()]:,} on {len(y_te)} customers)')

## 3. Let scikit-learn find it: `TunedThresholdClassifierCV`
Write the business scorer with `make_scorer`, wrap the model, and cross-validation picks the cutoff on the *training* folds — no peeking at the test set.

In [ ]:
profit_scorer = make_scorer(profit)
tuned = TunedThresholdClassifierCV(LogisticRegression(max_iter=2000), scoring=profit_scorer, cv=5).fit(X_tr, y_tr)
print(f'CV-tuned threshold: {tuned.best_threshold_:.3f}')
for name, m in [('threshold 0.50', base), ('tuned', tuned)]:
    pr = m.predict(X_te); print(f'{name:15s} profit ${profit(y_te, pr):>8,}   matrix {confusion_matrix(y_te, pr).tolist()}')
# When the business hands you the cutoff:
fixed = FixedThresholdClassifier(base, threshold=0.10).fit(X_tr, y_tr)
print(f'fixed 0.10       profit ${profit(y_te, fixed.predict(X_te)):>8,}')

The model did not change. Not one coefficient. We wrote down what each mistake costs and the arithmetic moved the cutoff.

## 4. Calibration: does 0.8 mean 80%?
AUC measures *ranking*. Calibration asks whether a score of 0.8 corresponds to an 80% churn rate. Logistic regression tends to sit on the diagonal; forests and boosting often do not — and every dollar in the profit curve was computed from those probabilities.

In [ ]:
rf = RandomForestClassifier(300, min_samples_leaf=2, random_state=835, n_jobs=-1).fit(X_tr, y_tr)
gbm = HistGradientBoostingClassifier(random_state=835).fit(X_tr, y_tr)
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, m in [('logistic', base), ('random forest', rf), ('gradient boosting', gbm)]:
    CalibrationDisplay.from_estimator(m, X_te, y_te, n_bins=10, name=name, ax=ax)
    print(f'{name:18s} AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}   Brier {brier_score_loss(y_te, m.predict_proba(X_te)[:, 1]):.4f}')
plt.show()

In [ ]:
# Repair with CalibratedClassifierCV (fit on training folds; isotonic for plenty of data, sigmoid for little)
rf_cal = CalibratedClassifierCV(RandomForestClassifier(300, min_samples_leaf=2, random_state=835, n_jobs=-1), method='isotonic', cv=5).fit(X_tr, y_tr)
print(f'random forest calibrated: AUC {roc_auc_score(y_te, rf_cal.predict_proba(X_te)[:, 1]):.3f}   Brier {brier_score_loss(y_te, rf_cal.predict_proba(X_te)[:, 1]):.4f}')
fig, ax = plt.subplots(figsize=(5.5, 5)); CalibrationDisplay.from_estimator(rf, X_te, y_te, n_bins=10, name='RF raw', ax=ax); CalibrationDisplay.from_estimator(rf_cal, X_te, y_te, n_bins=10, name='RF isotonic', ax=ax); plt.show()

## 5. Lift, gain, and targeting under a budget
When you can only contact 10%, the question is *who are the best 141?* Sort by score, contact from the top. The **gain chart** shows the share of churners reached; **lift** is the ratio to random.

In [ ]:
order = np.argsort(-p); yt = y_te.values[order]
cum = np.cumsum(yt) / yt.sum(); frac = np.arange(1, len(yt) + 1) / len(yt)
deciles = np.array([cum[int(len(yt) * d) - 1] for d in np.arange(0.1, 1.01, 0.1)])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(frac, cum, color='#2ee6c5', lw=2, label='model'); ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8, label='random'); ax[0].set_xlabel('share of customers contacted'); ax[0].set_ylabel('share of churners reached'); ax[0].set_title('gain chart'); ax[0].legend()
ax[1].bar(np.arange(1, 11), deciles / np.arange(0.1, 1.01, 0.1), color='#7c8cff'); ax[1].set_xlabel('decile'); ax[1].set_ylabel('cumulative lift'); ax[1].set_title('lift by decile')
plt.tight_layout(); plt.show()
print(f'top 10% captures {deciles[0]:.1%} of churners → lift {deciles[0]/0.1:.2f}')

In [ ]:
# Budget: contact the top-k, whatever the threshold
for k in [100, 200, 300, 500]:
    idx = order[:k]; tp = y_te.values[idx].sum()
    print(f'top {k:3d}: {tp:3d} churners caught, profit ${SAVE*tp - OFFER*k:>7,}, precision {tp/k:.2f}')

## 6. The honest caveat: churn ≠ uplift
A churn model finds customers **likely to leave**, not customers **an offer will persuade**. Some would have stayed anyway (wasted offer); some leave regardless (lost cause); some are *provoked* by contact (sleeping dogs). **Uplift modeling** estimates the treatment effect from an A/B test — treatment vs. control — and targets only the *persuadables*. Read Wayfair's explainer on the session page.

In [ ]:
# A five-line simulation of why: same churn scores, but the offer only changes behavior for a subset
rng = np.random.default_rng(835)
persuadable = rng.random(len(p)) < 0.35                     # only 35% of churners can be saved by the offer
val = np.where(persuadable, SAVE, 0)                         # value of catching each true churner
def true_profit(mask): return (val * ((y_te.values == 1) & mask)).sum() - OFFER * mask.sum()
print(f'naive expectation at tuned threshold: ${profit(y_te, tuned.predict(X_te)):,}   if only persuadables respond: ${true_profit(tuned.predict(X_te).astype(bool)):,}')

## 7. Midterm Model Competition — launch
Teams of 2–3. Hidden test set of hotel bookings; predict `is_canceled`. Public leaderboard on half the test set, private on the other half. **Closes Sun Nov 8, 11:59 PM.** Reveal + 3-minute "what worked" talks Nov 9. Graded 50% private-leaderboard band, 50% a one-page memo. Invitation link on Canvas.

## 8. Your turn
1. **Cost sensitivity.** Recompute the tuned threshold with OFFER = 200. Explain the direction of the move in one sentence.
2. **Calibrate a model that needs it.** Refit gradient boosting with `class_weight='balanced'` — the standard imbalance advice from Session 5 — and compare AUC and Brier score before and after `CalibratedClassifierCV`. Then do the same for the default `gbm`. Which one needed it, and why?
3. **Budget memo.** Marketing can afford 250 contacts. Which customers, what profit, and what would you tell them about the 251st customer?

In [ ]:
# Your turn — work here

## What we learned tonight
- **A model produces a score; a business produces a decision.** Break-even threshold p* = cost / (cost + value).
- **AUC is ranking; calibration is meaning.** Check the curve; `CalibratedClassifierCV` on held-out folds when trees bow away from the diagonal.
- **Under a budget the question is "who are the best k?"** — gain and lift charts. And a churn score is not an uplift score.

**HW3** (Nov 2): the credit-default threshold part. **Competition:** team formed and one submission by Nov 2.